# VGG16 Transfer Learning — Feature Extraction

**Cats vs Dogs • Binary Image Classification**

This notebook is part of a portfolio demonstrating **transfer learning with VGG16**.  
The goal is to compare pretrained feature extraction with task-specific fine-tuning while keeping the workflow reproducible and easy to inspect on GitHub.

> **Training note:** `EPOCHS = 2` is intentionally used for quick portfolio runs. Increase it later when doing a full experiment.

## Approach

1. Prepare the dataset
2. Load VGG16 pretrained on ImageNet
3. Build a lightweight task-specific classifier
4. Train for a short 2-epoch demonstration
5. Visualize training/validation performance
6. Run a sample prediction

The notebooks are designed to be readable as standalone project artifacts rather than raw classroom/Colab notebooks.

## 1. Configuration

The dataset is not committed to GitHub because image datasets can be large.  
Download the dataset separately and point `DATA_ROOT` to the extracted folder.

Expected structure:

```text
dogsvscats_dataset/
├── train/
│   ├── cats/
│   └── dogs/
└── test/
    ├── cats/
    └── dogs/
```

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Flatten
from tensorflow.keras.applications import VGG16
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array

SEED = 42
IMAGE_SIZE = (150, 150)
BATCH_SIZE = 32
EPOCHS = 2

# Change this path when running locally.
DATA_ROOT = Path("dogsvscats_dataset")

np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)
print("Dataset path:", DATA_ROOT.resolve())

### Optional: download the dataset in Google Colab

Run the following only if you are using Colab and have Kaggle credentials configured.  
For GitHub, keeping dataset download instructions separate from the modeling code makes the notebook cleaner and more portable.

In [ ]:
# Colab/Kaggle example:
# !pip -q install kaggle
# !mkdir -p dogsvscats_dataset
# !kaggle datasets download -d princelv84/dogsvscats -p dogsvscats_dataset
# !unzip -q dogsvscats_dataset/dogsvscats.zip -d dogsvscats_dataset

In [ ]:
TRAIN_DIR = DATA_ROOT / "train"
TEST_DIR = DATA_ROOT / "test"

assert TRAIN_DIR.exists(), f"Training directory not found: {TRAIN_DIR}"
assert TEST_DIR.exists(), f"Test directory not found: {TEST_DIR}"

print("Training directory:", TRAIN_DIR)
print("Test directory:", TEST_DIR)

## 2. Load the pretrained VGG16 backbone

VGG16 is used without its original ImageNet classification head.  
For **feature extraction**, the convolutional base remains frozen and only the new classifier layers learn from the Cats vs Dogs data.

In [ ]:
conv_base = VGG16(
    weights="imagenet",
    include_top=False,
    input_shape=(*IMAGE_SIZE, 3)
)

conv_base.trainable = False
conv_base.summary()

## 3. Build the classification model

The architecture follows the original project:

**VGG16 convolutional base → Flatten → Dense(256) → Sigmoid output**

In [ ]:
model = Sequential([
    conv_base,
    Flatten(),
    Dense(256, activation="relu"),
    Dense(1, activation="sigmoid")
])

model.summary()

## 4. Prepare image generators

Training images receive light augmentation. Validation/test images are only rescaled so that evaluation remains consistent.

In [ ]:
train_datagen = ImageDataGenerator(
    rescale=1./255,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True
)

test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary",
    seed=SEED
)

validation_generator = test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary",
    shuffle=False
)

print("Class mapping:", train_generator.class_indices)

## 5. Train the feature-extraction model

Only the newly added dense layers are trainable because the VGG16 backbone is frozen.

In [ ]:
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

history = model.fit(
    train_generator,
    epochs=EPOCHS,
    validation_data=validation_generator,
    verbose=1
)

## 6. Training curves

These plots provide a quick visual check of learning behavior during the 2-epoch portfolio run.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history.history["accuracy"], marker="o", label="Train")
axes[0].plot(history.history["val_accuracy"], marker="o", label="Validation")
axes[0].set_title("Accuracy")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Accuracy")
axes[0].legend()

axes[1].plot(history.history["loss"], marker="o", label="Train")
axes[1].plot(history.history["val_loss"], marker="o", label="Validation")
axes[1].set_title("Loss")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].legend()

plt.tight_layout()
plt.show()

## 7. Sample prediction

The class mapping is read from the generator rather than hard-coded, which keeps inference aligned with the dataset folders.

In [ ]:
class_names = {v: k for k, v in train_generator.class_indices.items()}

sample_path = next(TEST_DIR.glob("*/*.jpg"), None)
if sample_path is None:
    sample_path = next(TEST_DIR.glob("*/*.jpeg"), None)

if sample_path:
    img = load_img(sample_path, target_size=IMAGE_SIZE)
    img_array = img_to_array(img) / 255.0
    prediction = model.predict(np.expand_dims(img_array, axis=0), verbose=0)[0][0]

    predicted_index = int(prediction >= 0.5)
    predicted_label = class_names[predicted_index]

    print("Image:", sample_path)
    print(f"Predicted class: {predicted_label}")
    print(f"Dog probability: {prediction:.3f}")
else:
    print("No JPG/JPEG sample found in the test directory.")

## Key takeaway

**Feature extraction** uses VGG16 as a fixed visual representation and trains only a new task-specific classifier. This provides a fast baseline and establishes a reference point for the fine-tuning experiment in the companion notebook.